# Project 7: rijker sinds 2010, in welke valuta?

**De vraag.** Is de gemiddelde Surinamer sinds 2010 rijker geworden, gemeten in Amerikaanse dollars en in Surinaamse dollars (SRD)?

**De data.** Twee tabellen van de Wereldbank voor Suriname, allebei in lopende prijzen:

- Het bbp per inwoner in lopende Amerikaanse dollars (indicator NY.GDP.PCAP.CD), zoals de csv-download van de Wereldbank het ordent: één rij, met een kolom voor elk jaar van 1960 tot en met 2025.
- Het bbp per inwoner in lopende Surinaamse dollars (indicator NY.GDP.PCAP.CN), uit de API van de Wereldbank: één rij voor elk jaar van 2010 tot en met 2025.

De Wereldbank heeft beide voor het laatst bijgewerkt op 2026-10-08. De kopieën van de cursus zijn gedownload op 2026-11-06.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week telt de datacontroleur de rijen van elke tabel vóór en na elke koppeling, en toetst de AI-controleur elke koppeling die de AI schrijft aan die aantallen.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, twee grafieken, een notitie van ongeveer 150 woorden voor een gezin, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest beide tabellen uit de cursusmap, of rechtstreeks bij de Wereldbank als de bestanden er niet zijn.

In [ ]:
import io, json, urllib.request, zipfile
import pandas as pd

USD_URL = "https://api.worldbank.org/v2/country/SUR/indicator/NY.GDP.PCAP.CD?downloadformat=csv"
SRD_URL = ("https://api.worldbank.org/v2/country/SUR/indicator/NY.GDP.PCAP.CN"
           "?format=json&date=2010:2025&per_page=100")

# bbp per inwoner in Amerikaanse dollars: de csv-download van de Wereldbank, één kolom per jaar
try:
    usd_wide = pd.read_csv("https://santerre.ai/bigi-data/data/week07-gdp-per-person-usd.csv")
except FileNotFoundError:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(USD_URL).read()))
    name = [n for n in z.namelist() if n.startswith("API_")][0]
    usd_wide = pd.read_csv(z.open(name), skiprows=4).dropna(axis=1, how="all")

# bbp per inwoner in Surinaamse dollars: de API van de Wereldbank, één rij per jaar
try:
    srd = pd.read_csv("https://santerre.ai/bigi-data/data/week07-gdp-per-person-srd.csv")
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(SRD_URL))[1]
    srd = pd.DataFrame(
        [{"year": int(r["date"]), "gdp_srd": r["value"]} for r in raw if r["value"] is not None]
    ).sort_values("year").reset_index(drop=True)

print("usd_wide:", usd_wide.shape, "  srd:", srd.shape)

## TODO 1: beschrijf beide tabellen

Schrijf code die voor elke tabel het aantal rijen en kolommen, het eerste en laatste jaar, het aantal ontbrekende waarden en het type van het jaar afdrukt. In `usd_wide` zijn de jaren kolomnamen en geen waarden in een kolom, dus kijk naar `usd_wide.columns`.

In [ ]:
# TODO 1

## TODO 2: van breed naar lang

Maak een tabel met de naam `usd`, met twee kolommen, `year` en `gdp_usd`, en één rij voor elke jaarkolom van `usd_wide`. De methode `melt` doet dat.

Druk het type van `year` af in `usd` en in `srd`. Schrijf beide typen in jullie AI-logboek, want je hebt ze nodig bij TODO 3.

In [ ]:
# TODO 2
usd = None

In [ ]:
# CHECK 2
assert usd is not None, "usd is nog leeg"
assert {"year", "gdp_usd"} <= set(usd.columns), "usd heeft de kolommen year en gdp_usd nodig"
year_cols = [c for c in usd_wide.columns if str(c).isdigit()]
assert len(usd) == len(year_cols), "usd heeft één rij nodig voor elke jaarkolom van usd_wide"
assert usd["year"].astype(int).is_unique, "een jaar komt meer dan één keer voor"
print("CHECK 2 geslaagd:", len(usd), "rijen; year is", usd["year"].dtype, "in usd en", srd["year"].dtype, "in srd")

## TODO 3: koppel de twee tabellen (de AI-controle)

Maak een tabel met de naam `both`, met de kolommen `year`, `gdp_usd` en `gdp_srd`, één rij voor elk jaar van 2010 tot en met 2025, en `year` opgeslagen als geheel getal.

Schrijf eerst in jullie AI-logboek hoeveel rijen `usd` en `srd` hebben, en hoeveel rijen `both` hoort te hebben. Geef je AI-assistent daarna deze opdracht: *"Ik heb twee pandas-tabellen. usd heeft de kolommen year en gdp_usd, één rij per jaar van 1960 tot en met 2025. srd heeft de kolommen year en gdp_srd, één rij per jaar van 2010 tot en met 2025. Schrijf code die ze koppelt tot een tabel met de naam both, met één rij per jaar van 2010 tot en met 2025."*

Plak de code hieronder, voer hem uit en tel de rijen van `both`. Als pandas de koppeling weigert, laat de AI dan de foutmelding zien en toets het volgende antwoord op dezelfde manier. Noteer elke poging in het AI-logboek: de rijen die je verwachtte, de rijen die je kreeg, en elk jaar dat verloren ging of dubbel voorkwam.

In [ ]:
# TODO 3
both = None

In [ ]:
# CHECK 3
assert both is not None, "both is nog leeg"
assert pd.api.types.is_integer_dtype(both["year"]), "year in both moet een geheel getal zijn, geen tekst"
assert both["year"].is_unique, "een jaar komt meer dan één keer voor"
assert sorted(both["year"]) == sorted(srd["year"].astype(int)), "both moet precies de jaren van de SRD-tabel hebben"
assert both[["gdp_usd", "gdp_srd"]].notna().all().all(), "er ontbreken waarden"
u = usd_wide.iloc[0]
s = srd.assign(year=srd["year"].astype(int)).set_index("year")["gdp_srd"]
assert all(abs(r.gdp_usd - float(u[str(r.year)])) < 1e-6 and abs(r.gdp_srd - s[r.year]) < 1e-6
           for r in both.itertuples()), "een waarde staat naast het verkeerde jaar"
print("CHECK 3 geslaagd:", len(both), "rijen, van", both["year"].min(), "tot en met", both["year"].max())

## TODO 4: hoeveel keer zo groot?

Sla in `times_usd` op hoeveel keer zo groot het bbp per inwoner in Amerikaanse dollars in 2025 was als in 2010. Sla hetzelfde voor Surinaamse dollars op in `times_srd`.

In [ ]:
# TODO 4
times_usd = None
times_srd = None

In [ ]:
# CHECK 4
b = both.set_index("year")
assert times_usd is not None and times_srd is not None, "times_usd of times_srd is nog leeg"
assert abs(times_usd - b.loc[2025, "gdp_usd"] / b.loc[2010, "gdp_usd"]) < 0.01
assert abs(times_srd - b.loc[2025, "gdp_srd"] / b.loc[2010, "gdp_srd"]) < 0.01
print("CHECK 4 geslaagd:", round(times_usd, 2), "in US$ en", round(times_srd, 2), "in SRD")

## TODO 5: de wisselkoers in de cijfers

Beide tabellen meten hetzelfde in twee valuta's. Als je het SRD-cijfer deelt door het dollarcijfer, krijg je daarom het aantal SRD dat de Wereldbank in dat jaar voor één Amerikaanse dollar rekende. Voeg dit aan `both` toe als kolom met de naam `srd_per_usd`. Sla in `times_rate` op hoeveel keer zo groot die in 2025 was als in 2010.

Druk daarna `times_usd * times_rate` af naast `times_srd`. Je notitie moet in één zin uitleggen waarom de twee overeenkomen.

In [ ]:
# TODO 5
times_rate = None

In [ ]:
# CHECK 5
assert "srd_per_usd" in both.columns, "both heeft geen kolom srd_per_usd"
b = both.set_index("year")
assert (abs(b["srd_per_usd"] - b["gdp_srd"] / b["gdp_usd"]) < 1e-9).all()
assert times_rate is not None, "times_rate is nog leeg"
assert abs(times_rate - b.loc[2025, "srd_per_usd"] / b.loc[2010, "srd_per_usd"]) < 0.01
print("CHECK 5 geslaagd:", round(times_rate, 2))

## TODO 6: de jaren die daalden

Sla in `usd_down_years` een lijst op van de jaren waarin het bbp per inwoner in Amerikaanse dollars lager was dan het jaar ervoor. Sla hetzelfde voor Surinaamse dollars op in `srd_down_years`. Een lijst mag leeg zijn.

In [ ]:
# TODO 6
usd_down_years = None
srd_down_years = None

In [ ]:
# CHECK 6
assert usd_down_years is not None and srd_down_years is not None, "een lijst is nog leeg"
b = both.set_index("year").sort_index()
exp_usd = [int(y) for y in b.index[1:] if b.loc[y, "gdp_usd"] < b.loc[y - 1, "gdp_usd"]]
exp_srd = [int(y) for y in b.index[1:] if b.loc[y, "gdp_srd"] < b.loc[y - 1, "gdp_srd"]]
assert sorted(int(y) for y in usd_down_years) == exp_usd, "usd_down_years klopt niet"
assert sorted(int(y) for y in srd_down_years) == exp_srd, "srd_down_years klopt niet"
print("CHECK 6 geslaagd:", exp_usd, "in US$ en", exp_srd, "in SRD")

## TODO 7: twee grafieken naast elkaar

Teken het bbp per inwoner per jaar als twee lijngrafieken naast elkaar, één in Amerikaanse dollars en één in Surinaamse dollars. Geef elke grafiek een titel, benoem beide assen, laat beide waarde-assen bij nul beginnen en noem de Wereldbank als bron.

In [ ]:
# TODO 7

## TODO 8: de notitie voor een gezin

Schrijf in een tekstcel hieronder ongeveer 150 woorden voor een gezin dat vraagt of de gemiddelde inwoner van Suriname rijker is dan in 2010. Antwoord in beide valuta's met de getallen die je hebt berekend, leg uit waarom de twee antwoorden verschillen, noem de bron en zeg één ding dat de data hun niet kan vertellen.